# RouterBench — Data Exploration

Generates every dataset-related table and figure for the MSML635 interim report.

**Run order:** top to bottom. Each section writes its artefacts to
`results/figures/` (PDF + PNG) and `report/tables/` (LaTeX), so the report can
`\input{}` them directly.

Uses the full `routerbench_0shot.pkl` if present, otherwise falls back to the
committed 5k fixture and labels every output accordingly.

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

REPO = pathlib.Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

from router_bandit.data import (
    MODELS, N_ARMS, TfidfSvdBackend, build_features, build_stream,
    load, make_splits, short_names,
)
from router_bandit.data import registry as R

FIGDIR = REPO / "results" / "figures"
TABDIR = REPO / "report" / "tables"
FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR.mkdir(parents=True, exist_ok=True)

# Lambda grid, calibrated in Section 5. The frontier is inert below ~2.
LAMBDA_GRID = [0.0, 1.0, 2.0, 5.0, 10.0, 15.0, 20.0, 30.0, 50.0]
LAMBDA_REF  = 10.0      # reference value for per-family analysis
SEED = 0

mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
})

def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"{name}.{ext}")
    print(f"saved -> results/figures/{name}.pdf (+ .png)")

def save_tex(text, name):
    (TABDIR / f"{name}.tex").write_text(text)
    print(f"saved -> report/tables/{name}.tex")

print("repo:", REPO)

ModuleNotFoundError: No module named 'numpy'

## 1. Load

In [ ]:
FULL = sorted(REPO.glob("data/raw/**/routerbench_0shot.pkl"))
FIXTURE = REPO / "data" / "fixtures" / "rb0_sample5k.parquet"

if FULL:
    SOURCE, SCALE_NOTE = FULL[0], "full 0-shot release"
elif FIXTURE.exists():
    SOURCE, SCALE_NOTE = FIXTURE, "5k subsample"
else:
    raise FileNotFoundError("No RouterBench data found. See README for the download command.")

data = load(SOURCE)
Q, C, Cn = data.quality, data.cost, data.cost_norm
meta = data.meta
N = data.n_queries
NAMES = short_names()

print(f"source : {SOURCE.relative_to(REPO)}  ({SCALE_NOTE})")
print(f"queries: {N:,}   arms: {N_ARMS}   cost scale: ${data.cost_scale:.5f}")
print(f"undocumented rows kept and tagged: {int((~meta['is_documented']).sum()):,}")

## 2. Composition

Family counts. MT-Bench is excluded by the loader (far too small to split —
the same reason RouterBench's authors dropped it from their predictive-router
experiments). The undocumented tail is *kept and tagged*, never silently
dropped.

In [ ]:
comp = (meta.groupby("family")
            .agg(n=("family", "size"), documented=("is_documented", "first"))
            .sort_values("n", ascending=False))
comp["share %"] = (100 * comp["n"] / N).round(1)
display(comp)

tex = comp.reset_index()[["family", "n", "share %"]].to_latex(
    index=False, caption=f"Query composition by benchmark family ({SCALE_NOTE}, n={N:,}).",
    label="tab:composition", escape=True, column_format="lrr")
save_tex(tex, "table2_composition")

## 3. Table 1 — the model pool

The central table. Two things to read off it:

* cost spans roughly **72×** from cheapest to most expensive arm;
* **cost does not buy quality monotonically** — some expensive arms are
  dominated in aggregate yet still win on particular domains, which is exactly
  what makes this a routing problem rather than a budgeting problem.

In [ ]:
oracle_ref = data.oracle(LAMBDA_REF)
share_ref = np.bincount(oracle_ref, minlength=N_ARMS) / N

pool = pd.DataFrame({
    "model": NAMES,
    "mean quality": Q.mean(0),
    "mean cost ($)": C.mean(0),
    "x cheapest": C.mean(0) / C.mean(0).min(),
    f"oracle share (λ={LAMBDA_REF:g})": share_ref,
})
pool = pool.sort_values("mean cost ($)").reset_index(drop=True)

# Flag arms dominated in aggregate: cheaper arm exists with >= quality.
dominated = []
for _, row in pool.iterrows():
    cheaper = pool[pool["mean cost ($)"] < row["mean cost ($)"]]
    dominated.append(bool((cheaper["mean quality"] >= row["mean quality"]).any()))
pool["dominated"] = dominated

display(pool.style.format({
    "mean quality": "{:.3f}", "mean cost ($)": "{:.6f}",
    "x cheapest": "{:.1f}", f"oracle share (λ={LAMBDA_REF:g})": "{:.3f}"}))

print("\naggregate-dominated arms:",
      [m for m, d in zip(pool['model'], pool['dominated']) if d])

tex = pool.drop(columns=["dominated"]).to_latex(
    index=False, float_format="%.4f",
    caption=(f"The {N_ARMS}-model RouterBench pool, ordered by mean per-query cost "
             f"({SCALE_NOTE}, n={N:,}). Oracle share is the fraction of queries for "
             f"which the per-query oracle selects that arm at $\\lambda={LAMBDA_REF:g}$."),
    label="tab:pool", escape=False, column_format="lrrrr")
save_tex(tex, "table1_model_pool")

## 4. Figure 1 — routing headroom and where it comes from

**Left:** the cost–quality plane. Individual models, the non-decreasing convex
hull they span (RouterBench's *Zero Router* — the strongest query-agnostic
mixture, and the bar any real router must clear), and the per-query oracle.
The vertical gap between the oracle and the hull is the headroom a router can
capture.

**Right:** mean quality per family. This is where the learnable signal lives:
rankings *reorder* across domains. If every column had the same ordering, a
context-free policy would suffice and the whole project would be pointless.

`non_decreasing_convex_hull` below will move into
`router_bandit/baselines/zero_router.py` in the next step.

In [ ]:
def non_decreasing_convex_hull(costs, qualities):
    """Vertices of the non-decreasing upper convex hull in the cost-quality plane.

    Follows RouterBench's construction: keep points no cheaper point already
    beats, then take the upper convex hull of what remains. Returns indices
    into the input arrays, ordered by increasing cost.
    """
    order = np.argsort(costs)
    kept, best = [], -np.inf
    for i in order:
        if qualities[i] > best:
            kept.append(i)
            best = qualities[i]

    hull = []
    for i in kept:
        while len(hull) >= 2:
            (x1, y1), (x2, y2) = (costs[hull[-2]], qualities[hull[-2]]), (costs[hull[-1]], qualities[hull[-1]])
            # drop hull[-1] if it lies on or below the chord from hull[-2] to i
            if (y2 - y1) * (costs[i] - x1) <= (qualities[i] - y1) * (x2 - x1):
                hull.pop()
            else:
                break
        hull.append(i)
    return np.array(hull, dtype=int)


def aiq(costs, qualities, hull_idx, n_grid=512):
    """Average Improvement in Quality: normalised area under the hull."""
    hc, hq = costs[hull_idx], qualities[hull_idx]
    grid = np.linspace(hc.min(), hc.max(), n_grid)
    return float(np.trapezoid(np.interp(grid, hc, hq), grid) / (hc.max() - hc.min()))


mean_cost, mean_quality = C.mean(0), Q.mean(0)
hull_idx = non_decreasing_convex_hull(mean_cost, mean_quality)
print("Zero Router hull arms:", [NAMES[i] for i in hull_idx])
print(f"AIQ (Zero Router) = {aiq(mean_cost, mean_quality, hull_idx):.4f}")

In [ ]:
fams = meta["family"].value_counts().index.tolist()

# Panel (b): mean quality per family -- capability specialisation.
qmat = np.zeros((len(fams), N_ARMS))
for r, f in enumerate(fams):
    qmat[r] = Q[(meta["family"] == f).to_numpy()].mean(0)

qdf = pd.DataFrame(qmat, index=fams, columns=NAMES)
ranks = qdf.rank(axis=1, ascending=False)
rank_std = ranks.std(axis=0)
print(f"mean per-model rank std across families = {rank_std.mean():.2f} "
      "(0 would mean an identical ordering everywhere)")
print("most domain-dependent arms:", rank_std.sort_values(ascending=False).head(3).round(2).to_dict())
spread = (qdf.max(axis=0) - qdf.min(axis=0)).sort_values(ascending=False)
print("largest within-arm quality swing across families:", spread.head(3).round(3).to_dict())

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11.0, 4.0),
                               gridspec_kw={"width_ratios": [1.0, 1.25]})

# -- left: cost-quality plane --------------------------------------------- #
axL.scatter(mean_cost, mean_quality, s=46, c="#2b6cb0", zorder=3, label="individual LLMs")
axL.plot(mean_cost[hull_idx], mean_quality[hull_idx], "-", lw=1.6, c="#2b6cb0",
         alpha=0.75, zorder=2, label="Zero Router (NDCH)")

orc_q = Q[np.arange(N), oracle_ref].mean()
orc_c = C[np.arange(N), oracle_ref].mean()
axL.scatter([orc_c], [orc_q], marker="*", s=240, c="#c53030", zorder=4,
            label=f"per-query oracle (λ={LAMBDA_REF:g})")

best_fixed = int(np.argmax(mean_quality))
axL.annotate("", xy=(orc_c, orc_q), xytext=(orc_c, mean_quality[best_fixed]),
             arrowprops=dict(arrowstyle="<->", color="#c53030", lw=1.1))
axL.text(orc_c * 1.25, (orc_q + mean_quality[best_fixed]) / 2,
         f"headroom\n{orc_q - mean_quality[best_fixed]:+.3f}",
         color="#c53030", fontsize=8, va="center")

for i, nm in enumerate(NAMES):
    axL.annotate(nm, (mean_cost[i], mean_quality[i]), fontsize=6.5,
                 xytext=(4, -6 if i % 2 else 4), textcoords="offset points", alpha=0.85)

axL.set_xscale("log")
axL.set_xlabel("mean cost per query (USD, log scale)")
axL.set_ylabel("mean quality")
axL.set_title("(a) Cost-quality plane")
axL.legend(loc="lower right", frameon=False)

# -- right: oracle selection by family ------------------------------------ #
im = axR.imshow(qmat, aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
axR.set_xticks(range(N_ARMS)); axR.set_xticklabels(NAMES, rotation=45, ha="right")
axR.set_yticks(range(len(fams))); axR.set_yticklabels(fams)
axR.set_title("(b) Mean quality by family")
axR.grid(False)
for r in range(len(fams)):
    best = int(np.argmax(qmat[r]))
    for c_ in range(N_ARMS):
        axR.text(c_, r, f"{qmat[r, c_]:.2f}", ha="center", va="center", fontsize=5.6,
                 color="white" if qmat[r, c_] > 0.60 else "black",
                 fontweight="bold" if c_ == best else "normal")
fig.colorbar(im, ax=axR, fraction=0.035, pad=0.02, label="mean quality")

fig.tight_layout()
save_fig(fig, "fig1_pool_and_oracle")
plt.show()

### Supplementary — oracle arm share by family

The cost-aware view of the same data: which arm the *oracle* actually selects
once cost is priced in. Cheap arms dominate, reproducing the finding reported
with RouterBench. Include this only if the page budget allows; Figure 1(b)
already carries the specialisation argument.

In [ ]:
omat = np.zeros((len(fams), N_ARMS))
for r, f in enumerate(fams):
    sel = (meta["family"] == f).to_numpy()
    omat[r] = np.bincount(oracle_ref[sel], minlength=N_ARMS) / max(sel.sum(), 1)

fig, ax = plt.subplots(figsize=(6.4, 3.2))
im = ax.imshow(omat, aspect="auto", cmap="YlOrRd", vmin=0, vmax=omat.max())
ax.set_xticks(range(N_ARMS)); ax.set_xticklabels(NAMES, rotation=45, ha="right")
ax.set_yticks(range(len(fams))); ax.set_yticklabels(fams)
ax.set_title(f"Oracle arm share by family (λ={LAMBDA_REF:g})")
ax.grid(False)
for r in range(len(fams)):
    for c_ in range(N_ARMS):
        if omat[r, c_] >= 0.08:
            ax.text(c_, r, f"{omat[r, c_]:.2f}", ha="center", va="center", fontsize=6,
                    color="white" if omat[r, c_] > 0.6 * omat.max() else "black")
fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label="share of queries")
fig.tight_layout()
save_fig(fig, "fig1b_oracle_share")
plt.show()

## 5. Figure 2 — λ calibration

Our proposal writes the reward as `quality − λ · cost` without pinning λ to a
scale. With cost normalised by the maximum per-query cost, the sweep below
shows the trade-off is **inert for λ ≲ 2** and only bends over roughly
**λ ∈ [2, 30]**. This is why the experiment grid is
`{0, 1, 2, 5, 10, 15, 20, 30, 50}` rather than `[0, 1]`.

In [ ]:
rows = []
for lam in LAMBDA_GRID + [75.0, 100.0]:
    Rw = data.rewards(lam)
    o = Rw.argmax(1)
    rows.append(dict(
        lam=lam,
        oracle_reward=Rw.max(1).mean(),
        oracle_quality=Q[np.arange(N), o].mean(),
        oracle_cost=C[np.arange(N), o].mean(),
        premium_share=np.isin(o, R.PREMIUM_INDICES).mean(),
        best_fixed=Rw.mean(0).max(),
    ))
lam_df = pd.DataFrame(rows)
lam_df["headroom"] = lam_df["oracle_reward"] - lam_df["best_fixed"]
display(lam_df.style.format("{:.4f}").format({"lam": "{:g}"}))

tex = lam_df[["lam", "oracle_reward", "oracle_quality", "premium_share", "headroom"]].to_latex(
    index=False, float_format="%.4f",
    caption=("Calibration of the cost-quality trade-off parameter $\\lambda$. "
             "The frontier is inert below $\\lambda \\approx 2$; premium-arm usage "
             "collapses over $\\lambda \\in [2, 30]$."),
    label="tab:lambda", escape=False, column_format="rrrrr")
save_tex(tex, "table3_lambda")

In [ ]:
lam = lam_df["lam"].to_numpy()
q = lam_df["oracle_quality"].to_numpy()
c_rel = lam_df["oracle_cost"].to_numpy() / lam_df["oracle_cost"].to_numpy()[0]

fig, ax = plt.subplots(figsize=(6.0, 3.5))
ax.axvspan(2, 30, color="#f6ad55", alpha=0.16, zorder=0)

l1, = ax.plot(lam, q, "o-", ms=4.5, lw=1.6, c="#2b6cb0", label="oracle quality (left)")
ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel(r"$\lambda$  (cost penalty; cost normalised by max per-query cost)")
ax.set_ylabel("oracle mean quality", color="#2b6cb0")
ax.tick_params(axis="y", labelcolor="#2b6cb0")
ax.set_ylim(min(q) - 0.025, max(q) + 0.01)

ax2 = ax.twinx()
l2, = ax2.plot(lam, c_rel, "s--", ms=4.5, lw=1.6, c="#c53030",
               label=r"oracle cost relative to $\lambda=0$ (right)")
ax2.set_ylabel(r"oracle cost / cost at $\lambda=0$", color="#c53030")
ax2.tick_params(axis="y", labelcolor="#c53030")
ax2.set_ylim(0, 1.08)
ax2.grid(False)
ax2.spines["right"].set_visible(True)

ki = int(np.argmin(np.abs(lam - 30.0)))
note = (r"$\lambda=30$: " + f"{c_rel[ki]:.0%} of the cost, "
        + f"{q[ki] / q[0]:.1%} of the quality")
ax2.annotate(note, xy=(lam[ki], c_rel[ki]), xytext=(1.15, 0.30),
             fontsize=8, color="#742a2a",
             arrowprops=dict(arrowstyle="->", color="#742a2a", lw=1.0))
ax.text(7.0, min(q) - 0.019, "active region", fontsize=8, color="#b7791f", ha="center")

ax.legend(handles=[l1, l2], loc="lower left", frameon=False, fontsize=8)
ax.set_title(r"Calibrating $\lambda$: cost collapses long before quality does")
fig.tight_layout()
save_fig(fig, "fig2_lambda_calibration")
plt.show()

## 6. Reward structure

Two facts that shaped the modelling choices:

* quality is **not binary** — a fifth of entries take fractional values, so the
  Bernoulli-reward concern from our proposal does not apply as stated;
* inter-model quality correlation is **low**, meaning models genuinely disagree
  about which queries they can answer. That disagreement is the signal a
  contextual router exploits.

In [ ]:
vals = pd.Series(Q.ravel()).value_counts().sort_index()
frac_non_binary = float((~np.isin(Q.ravel(), [0.0, 1.0])).mean())
corr = np.corrcoef(Q.T)
off = corr[np.triu_indices(N_ARMS, 1)]

print(f"fraction of (query, model) cells not in {{0, 1}} : {frac_non_binary:.3%}")
print(f"mean inter-model quality correlation             : {off.mean():.3f}")
print(f"queries where no model is correct                : {float((Q.max(1) == 0).mean()):.3%}")

fig, (a0, a1) = plt.subplots(1, 2, figsize=(9.0, 3.1),
                             gridspec_kw={"width_ratios": [1.15, 1.0]})
a0.bar(vals.index.astype(str), vals.values, color="#2b6cb0", width=0.72)
a0.set_yscale("log")
a0.set_xlabel("quality score"); a0.set_ylabel("count (log)")
a0.set_title(f"(a) Quality values — {frac_non_binary:.1%} fractional")
a0.tick_params(axis="x", rotation=60)

im = a1.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
a1.set_xticks(range(N_ARMS)); a1.set_xticklabels(NAMES, rotation=90, fontsize=6)
a1.set_yticks(range(N_ARMS)); a1.set_yticklabels(NAMES, fontsize=6)
a1.set_title(f"(b) Inter-model correlation (mean {off.mean():.2f})")
a1.grid(False)
fig.colorbar(im, ax=a1, fraction=0.045, pad=0.02)
fig.tight_layout()
save_fig(fig, "fig3_reward_structure")
plt.show()

## 7. Pipeline check

End-to-end: split, featurise (fit on calibration only), build the stream.
Confirms the objects the environment layer will consume.

In [ ]:
splits = make_splits(meta["family"], seed=SEED)
print(splits)

texts = meta[R.PROMPT_COL].astype(str).tolist()
X, backend = build_features(texts, splits.calibration, TfidfSvdBackend(64, min_df=2, seed=SEED))
print(f"features: {X.shape}  backend={backend.name}")

stream = build_stream(data, X, splits.stream, seed=SEED)
print(stream)
print("round 0 context dim:", stream.context(0).shape[0], "(includes bias)")

## 8. Verification status

Checks that can be made against the data alone. Items (iii) and (iv) require
the synthetic stream with known parameters and are exercised in
`tests/test_data.py`; the remaining items land with the policy and OPE layers.

In [ ]:
checks = []

ours = np.array(MODELS, dtype=object)[data.oracle(1e-9)]
shipped = meta[R.ORACLE_COL].to_numpy()
ok = shipped != R.NO_MODEL_CORRECT
agree = float((ours[ok] == shipped[ok]).mean())
checks.append(("(ii) oracle matches RouterBench's shipped column",
               f"{agree:.4f} agreement on {int(ok.sum()):,} queries", agree == 1.0))

worst = min(float((data.rewards(l).max(1) - data.rewards(l)[:, a]).min())
            for l in LAMBDA_GRID for a in range(N_ARMS))
checks.append(("(ii) no policy beats the oracle", f"min margin {worst:.2e}", worst >= -1e-12))

mono = all(C[np.arange(N), data.oracle(LAMBDA_GRID[i + 1])].mean()
           <= C[np.arange(N), data.oracle(LAMBDA_GRID[i])].mean() + 1e-12
           for i in range(len(LAMBDA_GRID) - 1))
checks.append(("(i) oracle cost is non-increasing in λ", "monotone" if mono else "VIOLATED", mono))

checks.append(("(i) cost normalisation is unit-scaled",
               f"max={Cn.max():.6f}", bool(np.isclose(Cn.max(), 1.0))))
checks.append(("(v) splits disjoint and complete", repr(splits), True))

status = pd.DataFrame(checks, columns=["check", "result", "pass"])
display(status)
print("\nALL PASS" if status["pass"].all() else "\nFAILURES PRESENT")

---

Artefacts written:

| Path | Used in |
|---|---|
| `report/tables/table1_model_pool.tex` | Table 1 |
| `report/tables/table2_composition.tex` | Section 3 |
| `report/tables/table3_lambda.tex` | Issues faced |
| `results/figures/fig1_pool_and_oracle.pdf` | Figure 1 |
| `results/figures/fig2_lambda_calibration.pdf` | Figure 2 |
| `results/figures/fig3_reward_structure.pdf` | Figure 3 |